# Chapter 11: one page of losses

Builds the nine row ledger and both waterfalls. The cellar row comes from the minimum dataset (the quarter's excess, times four). The other rows come from `loss_drivers.csv`: a quantity and a value per unit for each loss, as worked in its chapter. Any row without a sum is flagged.

Synthetic Kestrel Brewing data. No real brewery's numbers. Book rates: USD 1 = INR 90; EUR 1 = USD 1.10.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

drivers = pd.read_csv("../data/loss_drivers.csv")
df = pd.read_csv("../data/minimum_dataset.csv")
w = df[df.stage != "cast"].pivot(index="batch_id", columns="stage", values=["volume_in_hl", "volume_out_hl", "heel_hl"])
cast = w.volume_in_hl.fermentation.sum()
steps = [
    (w.volume_in_hl.fermentation - w.volume_out_hl.fermentation).sum(),
    (w.volume_in_hl.conditioning - w.volume_out_hl.conditioning - w.heel_hl.conditioning).sum(),
    w.heel_hl.conditioning.sum(),
    (w.volume_out_hl.conditioning - w.volume_in_hl["filter"]).sum(),
]
normal = [1.00, 0.32, 0.30, 0.30]
excess_q = sum(max(0, round(s - cast * n / 100)) for s, n in zip(steps, normal))
drivers.loc[drivers.loss == "Cellar", "quantity"] = 4 * excess_q
print(f"Cellar excess: {excess_q} hL a quarter, {4 * excess_q} hL a year")

In [ ]:
d = drivers.copy()
d["replacement_usd"] = (d.quantity * d.replacement_usd_per_unit + d.material_usd).round(0)
d["upper_bound_usd"] = (d.quantity * d.upper_bound_usd_per_unit).round(0)  # 143 a packed hL already holds the can
missing = d[d.replacement_usd.isna()]
for _, r in missing.iterrows():
    print(f"Row {r.row} ({r.loss}) has no sum: {r.unit}")
ledger = d.dropna(subset=["replacement_usd"])
print(d[["row", "loss", "quantity", "unit", "replacement_usd", "upper_bound_usd", "from_chapter"]].to_string(index=False))
rep, ub = ledger.replacement_usd.sum(), ledger.upper_bound_usd.sum()
print(f"Total: USD {rep:,.0f} replacement cost (INR {rep * 90:,.0f}; EUR {rep / 1.1:,.0f}), {rep / 100000:.2f} a hL; upper bound USD {ub:,.0f}")

## Does the sum of the parts match finance's gap? (chapter 11.3)

In [ ]:
FINANCE_VARIANCE = 128000  # finance's yield and usage variance for the year (synthetic)
materials_rows = ledger[ledger.row <= 5].replacement_usd.sum()
gap = FINANCE_VARIANCE - materials_rows
print(f"Rows 1 to 5: {materials_rows:,.0f}; finance {FINANCE_VARIANCE:,}; not yet explained {gap:,.0f} ({100 * gap / FINANCE_VARIANCE:.0f} percent)")
print("Above 10 percent: say so on the page." if gap / FINANCE_VARIANCE > 0.10 else "Under 10 percent: send it.")
beer = ledger[ledger.row.between(2, 5)]
print(f"Beer rows: {beer.quantity.sum():,.0f} hL; replacement {beer.replacement_usd.sum():,.0f}; upper bound {beer.upper_bound_usd.sum():,.0f}; at price {beer.quantity.sum() * 150:,.0f}")
top3 = ledger.nlargest(3, "replacement_usd")
print(f"Top three: {', '.join(top3.loss)} = {top3.replacement_usd.sum():,.0f}, {100 * top3.replacement_usd.sum() / rep:.1f} percent")

In [ ]:
def waterfall(ax, names, values, extra=None, title=""):
    run = 0
    for i, (n, v) in enumerate(zip(names, values)):
        ax.barh(i, v, left=run, color="#2f7f72")
        ax.text(run + v + 800, i, f"{v:,.0f}", va="center", fontsize=8)
        run += v
    k = len(names)
    if extra:
        ax.barh(k, extra[1], left=run, color="grey")
        ax.text(run + extra[1] + 800, k, f"{extra[1]:,.0f}", va="center", fontsize=8)
        names = list(names) + [extra[0]]
        k += 1
    ax.barh(k, run, color="#0b4f45")
    ax.text(run + 800, k, f"{run:,.0f}", va="center", fontsize=8)
    ax.set_yticks(range(k + 1), list(names) + ["Total" if not extra else "Losses plus gap"])
    ax.invert_yaxis()
    ax.set_title(title)


fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
waterfall(a1, ledger.loss, ledger.replacement_usd, title=f"Figure 11.1: by stage, USD a year (upper bound {ub:,.0f}, never added)")
s = ledger.sort_values("replacement_usd", ascending=False)
waterfall(a2, s.loss, s.replacement_usd, extra=("Not yet explained", gap), title="Figure 11.2: sorted, with the gap in grey")
plt.tight_layout()
plt.show()